# STEP 4-B — 증강 빠른 스윕

## 멘토링 피드백 (2026-08-21)

> "augmentation 기법들도 다양하게 시도"
> "multi scale 로 augmentation 할 때 scale 변화 범위의 **폭을 줄여보는** 시도를 해도 좋을듯"
> "augmentation 은 무조건 학습 중 실시간으로 랜덤. **albumentations**"
> "초반에는 최대한 빠르게 **자동으로 시도해볼 수 있는 것들** 시도하기"

마지막 항목은 **작업의 종류**에 대한 이야기입니다 — 실행 시간을 줄이라는 게 아니라,
**수작업 데이터 큐레이션을 초반에 하지 말라**는 뜻입니다:

| 초반에 **하지 말 것** (수작업) | 초반에 **할 것** (자동) |
|---|---|
| 데이터를 직접 보며 손으로 처리 | 증강 프리셋 비교 |
| 헷갈리는 케이스만 골라 개별 처리 | 해상도·백본 비교 |
| 학습 안 되는 데이터 과감히 삭제 | 임계값·보정 조정 |
| 클래스를 합치거나 재정의 | — 코드로 결론이 나는 것들 |

수작업 큐레이션은 비싸고 되돌리기 어렵습니다. 게다가 **자동 실험으로 조건이 바뀌면
어떤 데이터가 문제인지도 같이 바뀝니다.** 기준이 흔들리는 상태에서 손으로 고르면
헛수고가 됩니다. 그래서 5번("정성평가는 보통 후반부")과 짝을 이룹니다.

이 노트북은 그 "자동으로 시도할 수 있는 것" 에 해당합니다.

## 8종을 다 풀로 돌리지는 않습니다 (우리 판단)

프리셋 7종을 각각 풀 데이터·30에폭으로 돌리면 **10시간**이 넘습니다. Kaggle 한 세션에
안 들어갑니다. 그래서 **스크리닝 → 확정** 두 단계로 나눕니다:

```
이 노트북:  7종 × 55% 서브셋 × 12에폭  ≈ 3시간   → 순위를 매기고
다음 실행:  상위 2종 × 풀데이터 × 30에폭 ≈ 3시간   → 확정
```

⚠️ **이건 멘토 피드백이 아니라 GPU 예산 때문입니다.**
그리고 **서브셋 순위가 풀 데이터 순위와 같다는 보장은 없습니다.**
이 노트북은 **후보를 줄이는 용도**이고, 채택은 반드시 풀 스케일로 확인합니다.

## albumentations 로 갈아탔습니다

`bench.report` 실측에서 **증강·변환이 18.0ms 로 파이프라인 병목**이었습니다
(GPU 는 112 img/s 인데 실제 처리량이 90 img/s — 데이터를 기다리며 놀고 있었습니다).

같은 변환을 두 라이브러리로 재본 결과:

| | 384px 한 장 |
|---|---:|
| torchvision (PIL) | 22.6 ms |
| **albumentations** (numpy+cv2) | **7.3 ms** |
| albumentations · 기법 많이 | 7.5 ms |

**3.1배 빠르고, 기법을 늘려도 거의 안 느려집니다.** 다만 입력이 빨라지면 GPU 가
병목이 되므로 실제 단축은 90 → 약 112 img/s (**1.24배**) 정도입니다.

> 증강이 **학습 중 실시간 랜덤**인 건 원래부터 그랬습니다
> (`SkinDataset.__getitem__` 에서 매 호출마다 적용 = 에폭마다 다른 난수).
> 미리 만들어 저장하는 방식이 아닙니다. 바뀐 건 라이브러리입니다.
>
> ⚠️ **검증 변환은 torchvision 그대로 뒀습니다.** 검증 전처리를 바꾸면
> 지금까지 쌓은 기준선(1단계 0.8192 / 2단계 0.5697)과 비교가 안 됩니다.

## 무엇을 비교하나

2단계만 돌립니다 (1단계는 배율 하락 9.1% 로 이미 목표 달성).

| 프리셋 | 가설 | 출처 |
|---|---|---|
| `default` | 비교 기준 | — |
| `narrow` | 폭을 좁히면 과제가 쉬워져 점수가 오를까 | **멘토 1번** |
| `narrower` | 더 좁히면 더 좋을까 (좁히기 축의 두 번째 점) | **멘토 1번** |
| `zoom_both` | 축소를 세게 가르치면 (224px 에서는 실패) | 기존 |
| `zoom_mild` | 축소를 완만하게 — 좁히기와 축소의 절충 | 우리 설계 |
| `shift` | 위치 교란(20.6% 하락)을 이동 증강으로 | 우리 실측 |
| `photometric` | 흐림·노이즈·JPEG — 촬영 조건 흉내 | 멘토 4번 + 우리 실측 |

### 좁히기 축에 점을 **두 개** 찍은 이유

`rrc_scale` 하한은 "면적의 몇 % 이상을 남기고 자를지" 입니다. 클수록 덜 자릅니다.

```
0.70          0.85        0.92      1.0
 │             │           │
default      narrow    narrower
```

한 점만 찍으면 "0.70보다 0.85가 낫다" 까지만 알고 **"더 좁혀야 하나"** 는 모릅니다.
그럼 또 한 판 돌려야 하죠. 두 점이면 모양이 보입니다:

| 결과 | 읽는 법 |
|---|---|
| 계속 좋아짐 | 더 좁혀야 함 |
| `narrow` 가 최고, `narrower` 는 다시 나빠짐 | 그 근처가 최적점 — 끝 |
| 셋 다 비슷 | 이 축은 상관없음. 다른 데를 보자 |

`photometric` 에는 근거가 하나 더 있습니다. 크롭 감사에서
**정상 사진 선명도 중앙값 39 vs 병변 271** 이 나왔습니다. 모델이 피부가 아니라
**화질**로 맞힐 여지가 있다는 뜻이라, 흐림을 훈련에 넣으면 그 지름길을 막는
효과도 기대할 수 있습니다.

## 판정 기준 (돌리기 **전에** 정해둡니다)

이 노트북은 **채택하지 않습니다.** 순위만 매기고 상위 2종을 고릅니다.

| 볼 것 | 기준 |
|---|---|
| 1순위 | **배율 하락**이 `default` 보다 잡음(±3%p)을 넘겨 줄었는가 |
| 2순위 | 그러면서 macro-F1 손실이 0.02 이내인가 |
| 참고 | 최악 조건이 축소에서 확대로 옮겨갔는가 (원인 진단) |

⏱️ 7종 × 약 24분 ≈ **2시간 50분**. Kaggle 은 `Save & Run All (Commit)`.


In [ ]:
# ── 0. 환경 준비 (Colab / Kaggle 공통) ──────────────────────────
# 이 셀 하나가 리포 동기화 → 패키지 설치 → 환경 감지까지 다 합니다.
# 리포를 직접 다운로드하거나 드라이브에 올릴 필요 없습니다.
# 다시 실행하면 항상 최신 코드로 맞춰집니다 (로컬 수정은 덮어씁니다).
import os, sys, subprocess

REPO   = "https://github.com/gayeoniee/deeplearning_test.git"
BRANCH = "main"
NAME   = "deeplearning_test"
_cwd   = os.getcwd()
if os.path.basename(_cwd) == NAME and os.path.isdir(os.path.join(_cwd, ".git")):
    DIR = _cwd            # 이미 리포 안에서 재실행 중 (중첩 clone 방지)
else:
    # ⚠️ Kaggle 을 먼저 봅니다. Kaggle 이미지에도 /content 가 있어서
    #    /content 를 먼저 보면 Kaggle 세션인데 /content 에 clone 합니다.
    BASE = ("/kaggle/working" if os.path.isdir("/kaggle/working")
            else "/content" if os.path.isdir("/content") else _cwd)
    DIR = os.path.join(BASE, NAME)

if os.path.isdir(os.path.join(DIR, ".git")):
    # 이미 받아둔 경우: 최신으로 강제 동기화 (shallow clone 에서도 안전)
    subprocess.run(["git", "-C", DIR, "fetch", "--depth", "1", "origin", BRANCH], check=False)
    subprocess.run(["git", "-C", DIR, "reset", "--hard", f"origin/{BRANCH}"], check=False)
else:
    subprocess.run(["git", "clone", "-b", BRANCH, "--depth", "1", REPO, DIR], check=True)

os.chdir(DIR)
if DIR not in sys.path:
    sys.path.insert(0, DIR)

# ⚠️ 중요: 이미 import 된 src.* 는 파이썬이 캐시하고 있어서
#    파일을 갱신해도 옛날 코드가 그대로 쓰입니다. 캐시를 비웁니다.
for _m in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[_m]

print("작업 디렉터리:", os.getcwd())
print("코드 버전   :", subprocess.run(["git", "-C", DIR, "log", "--oneline", "-1"],
                                      capture_output=True, text=True).stdout.strip())

# 패키지 설치는 **uv 로 통일**합니다 (pip 보다 훨씬 빠릅니다).
# ⚠️ Colab/Kaggle 이미지에는 uv 가 없어서, uv 자체만 pip 로 한 번 받습니다.
#    --system = 가상환경을 새로 만들지 않고 이미 있는 파이썬에 그대로 설치.
#    (torch/numpy/pandas 는 이미 깔려 있으므로 여기서 안 건드립니다)
# albumentations 는 import 할 때마다 PyPI 에 버전 확인 요청을 보냅니다.
# Kaggle 은 외부 네트워크가 막혀 있어 타임아웃(2초)만 기다리다 끝납니다 — 꺼둡니다.
os.environ["NO_ALBUMENTATIONS_UPDATE"] = "1"

_PKGS = ["timm", "imagehash", "pyarrow", "grad-cam", "albumentations"]
_ok = False
if subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"],
                  check=False).returncode == 0:
    _ok = subprocess.run([sys.executable, "-m", "uv", "pip", "install", "-q",
                          "--system", *_PKGS], check=False).returncode == 0
if not _ok:
    print("[env] uv 로 설치하지 못해 pip 으로 대체합니다")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_PKGS], check=False)

# 한글 그래프 폰트 (Colab 기본에는 한글이 없어 □ 로 나옵니다)
_font = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(_font):
    subprocess.run(["apt-get", "install", "-y", "-qq", "fonts-nanum"], check=False)
try:
    import matplotlib.pyplot as plt, matplotlib.font_manager as fm
    fm.fontManager.addfont(_font)
    plt.rcParams["font.family"] = "NanumGothic"
    plt.rcParams["axes.unicode_minus"] = False
except Exception:
    pass

MY_NOTEBOOK_VERSION = "2026-08-22.9"   # ★ 이 셀(=이 .ipynb)의 버전

from src import env
from src.config import CFG, CLASSES, CLASS_KO
E = env.describe()
env.set_seed(42)

# 환경 판정이 이상하면(예: Kaggle 인데 colab 이라고 나오면) 근거를 봅니다
if E.env != "local":
    env.diagnose()

# ⚠️ 노트북 셀은 git pull 로 갱신되지 않습니다 (src/ 만 최신이 됩니다).
#    낡은 .ipynb 를 몇 시간 돌리고 나서 알게 되면 늦으므로 지금 확인합니다.
from src.config import NOTEBOOK_VERSION as _repo_nb
if MY_NOTEBOOK_VERSION != _repo_nb:
    print("\n" + "!" * 62)
    print(f"⚠️ 이 노트북이 낡았습니다 — 내 셀 {MY_NOTEBOOK_VERSION} / 리포 {_repo_nb}")
    print("   src/ 는 최신이지만 **셀 내용은 예전 것**입니다.")
    print("   GitHub 에서 notebooks/*.ipynb 를 다시 받아 Import 하세요:")
    print("   Kaggle → File → Import Notebook / Colab → 파일 → 노트 업로드")
    print("!" * 62 + "\n")
else:
    print(f"[nb] 노트북 최신 ({_repo_nb})")


---
## 1. 데이터 붙이기


In [ ]:
# Drive 마운트는 **진짜 Colab VM** 에서만 시도합니다.
# ⚠️ Kaggle 에도 google.colab 패키지와 /content 가 있어서, 환경 판정을 잘못하면
#    Kaggle 에서 drive.mount() 를 부르고 NotImplementedError 로 죽습니다.
if env.can_mount_drive():
    env.mount_drive()
else:
    print(f"[env] {E.env} — Drive 마운트 없이 진행합니다")

# 전처리 결과를 붙입니다. 두 가지 형태를 다 받습니다:
#   · Colab  : Drive 의 dogskin_prepared.zip → 로컬 디스크로 해제
#   · Kaggle : /kaggle/input/<데이터셋>/crops,manifests → 링크만 연결
#              (Kaggle 은 업로드한 zip 을 알아서 풀어둡니다. 복사하면 20GB 제한에 걸려요)
env.load_prepared()          # 경로를 직접 주려면: env.load_prepared("/kaggle/input/dogskin-prepared")

# ── 다른 환경에서 학습한 체크포인트 가져오기 (Colab → Kaggle 이주) ──────
#    Colab 에서 이미 학습을 끝냈다면, Drive 의 dogskin_work/checkpoints 를
#    Kaggle 데이터셋으로 올린 뒤 그 경로를 여기에 주세요.
#    가져온 실험은 '완료' 로 인식되어 학습 셀이 ⏭️ 로 건너뜁니다.
#
# train.import_checkpoints("/kaggle/input/dogskin-ckpt")

# 세션이 끊겨도 남는 저장소 확인
_persist = env.persist_root()
if _persist is None:
    print("\n🚨 세션 밖 저장소가 없습니다 — 지금 학습하면 끊길 때 체크포인트가 사라집니다.")
    print("   위 셀에서 Drive 마운트가 됐는지 확인하세요 (env.mount_drive()).")
else:
    print(f"\n✅ 중단 대비 저장소: {_persist}")
    if E.env == "kaggle":
        print("   ⚠️ Kaggle 은 세션이 끝나면 /kaggle/working 이 사라질 수 있습니다.")
        print("      · 짧게 확인만 할 때  : 그냥 진행 (세션 안에서는 이어받기가 됩니다)")
        print("      · 긴 학습을 돌릴 때  : 우측 상단 [Save Version] →")
        print("                             'Save & Run All (Commit)' 로 돌리세요.")
        print("                             브라우저를 닫아도 끝까지 돌고, 출력이 보존됩니다.")
        print("      · 설정에 Persistence 항목이 보이면 'Files' 로 켜두면 더 안전합니다")
    else:
        print("   매 에폭 체크포인트를 여기로 복사합니다. 세션이 끊기면 노트북을 처음부터")
        print("   다시 돌리세요 — 끝난 학습은 건너뛰고 끊긴 학습만 이어서 합니다.")

In [ ]:
import torch
from src import labels, split, crop, data, models, train, evaluate, stages, experiments, bench
from src.config import CLASSES, CFG, with_aug

env.require_gpu()
DEV = "cuda" if torch.cuda.is_available() else "cpu"

# ── 03 에서 확정된 설정 ────────────────────────────────────────
BEST_CROP    = "m1.5"   # 2단계 크롭
IMG_SIZE     = 384      # 해상도 실험에서 채택
SUBSET_FRAC  = 0.55     # ★ 스윕용 — 학습셋만 55%. 검증셋은 그대로 씁니다
EPOCHS       = 12       # ★ 스윕용 — 확정 실행은 25~30

# 03 의 풀 실행 실측 (표본 2,000장). 스윕은 데이터를 줄였으므로 점수가 이보다
# 낮게 나오는 게 정상입니다. **프리셋끼리만** 비교하세요.
FULL_384 = {"macro_f1": 0.5697, "scale_drop": 0.259, "worst_at": "배율(0.5x)"}

df = labels.load(env.work_root()/"manifests"/"manifest_final.parquet")
d = crop.switch_tag(df, BEST_CROP)
s2_all = stages.to_stage2(d)
split.verify(s2_all, fold=0, strict=True)      # 누수 재확인
print(f"2단계 {len(s2_all):,}행")

# albumentations 가 실제로 붙었는지 확인 (없으면 torchvision 으로 조용히 돌아갑니다)
_tf = data.build_transforms(CFG(img_size=IMG_SIZE), train=True)
print(f"학습 증강 백엔드: {type(_tf).__name__}"
      f"{'  ✅ albumentations' if 'Album' in type(_tf).__name__ else '  ⚠️ torchvision 폴백'}")


---
## 2. 처리량 확인


In [ ]:
# 증강이 실제로 얼마나 빨라졌는지 — 주장 말고 숫자로
_bcfg = CFG(model_name="resnet50", img_size=IMG_SIZE, balance_strategy="class_weight")
perf = bench.report(split.get_fold(s2_all, 0)[0], _bcfg, classes=CLASSES)


---
## 3. 프리셋 8종 스윕


In [ ]:
# 7종. zoom_shift(= zoom_mild + shift 라 결과가 예상됨)와
# kitchen_sink(나쁠 걸 알고 돌리는 대조군)를 빼고, 그 자리에 narrower 를 넣었습니다.
PRESETS = ("default", "narrow", "narrower", "zoom_both",
           "zoom_mild", "shift", "photometric")

runs = []
for preset in PRESETS:
    runs.append(experiments.train_and_measure(
        s2_all, stage=2, img_size=IMG_SIZE, crop_tag=BEST_CROP, device=DEV,
        epochs=EPOCHS, aug=preset, subset_frac=SUBSET_FRAC, n_robust=2000))

verdict = experiments.augmentation_report(runs, baseline="default")


---
## 4. 상위 2종 고르기


In [ ]:
# ★ 이 노트북은 채택하지 않습니다. 다음 풀 실행에 올릴 **상위 2종**만 고릅니다.
import json

base = next(r for r in runs if r["aug"] == "default")
cands = [r for r in runs if r["aug"] != "default" and r.get("scale_drop") is not None]

# 1순위 = 배율 하락이 작은 것. 단, macro-F1 을 0.02 넘게 잃으면 제외합니다.
ok = [r for r in cands if r["score"] >= base["score"] - 0.02]
ranked = sorted(ok or cands, key=lambda r: r["scale_drop"])
top2 = [r["aug"] for r in ranked[:2]]

print("=" * 62)
print(" STEP 4B 스윕 결과 (이 블록을 복사해서 공유하세요)")
print("=" * 62)
print(f"  서브셋 {SUBSET_FRAC:.0%} · {EPOCHS}에폭 · 학습 {base['n_train']:,}장\n")
print(f"  {'프리셋':<16}{'macro-F1':>10}{'배율하락':>10}{'최악조건':>12}")
for r in sorted(runs, key=lambda r: r.get("scale_drop", 9)):
    mark = " ←" if r["aug"] in top2 else ("  (기준)" if r["aug"] == "default" else "")
    print(f"  {r['aug']:<16}{r['score']:>10.4f}{r['scale_drop']:>9.1%}"
          f"{str(r.get('scale_worst_at','?')):>12}{mark}")
if not ok:
    print("\n  ⚠️ macro-F1 손실 0.02 이내인 후보가 없어 하락폭만으로 골랐습니다.")
print(f"\n  다음 풀 실행 후보: {', '.join(top2)}")
print(f"  (03 풀 실행 기준: macro-F1 {FULL_384['macro_f1']:.4f} / 하락 {FULL_384['scale_drop']:.1%})")
print("=" * 62)

W = env.work_root(); (W/"reports").mkdir(parents=True, exist_ok=True)
keep = ("stage","img_size","crop_tag","aug","subset_frac","n_train","exp_name","epochs",
        "batch_size","minutes","best_epoch","n_epochs","converged","score","score_name",
        "macro_f1","a6_recall","scale_drop","scale_worst","scale_worst_at")
(W/"reports"/"step4b_sweep.json").write_text(json.dumps({
    "subset_frac": SUBSET_FRAC, "epochs": EPOCHS, "img_size": IMG_SIZE,
    "full_384_reference": FULL_384, "top2": top2,
    "runs": [{k: r[k] for k in keep if k in r} for r in runs],
}, indent=2, ensure_ascii=False))
print(f"저장: {W/'reports'/'step4b_sweep.json'}")


---
## 다음 단계

이 노트북은 **후보를 줄였을 뿐입니다.** 상위 2종을 풀 데이터·30에폭으로 다시 돌려
확정합니다 (약 3시간). 거기서 배율 하락이 15% 안에 들어오면 04(백본 비교)로 갑니다.

### 순서 (멘토 피드백 5·6번)

**자동으로 되는 것을 다 짜낸 뒤에** 수작업으로 넘어갑니다.

| 순서 | 작업 | 성격 |
|---|---|---|
| 1 | 증강 확정 (이 노트북 → 풀 스케일) | 자동 |
| 2 | 백본 비교 `04` | 자동 |
| 3 | 임계값·온도 보정 `05` | 자동 (학습 없음) |
| 4 | 촬영 가이드 확정 — 배율별 표에서 **1x 가 최고** | 자동 (측정 끝남) |
| 5 | **정성평가** — 틀린 케이스를 직접 눈으로 | 수작업 |
| 6 | confusion 분석 → 클래스 재정의 판단 | 수작업 |

**5·6번을 지금 하면 안 됩니다.** A1 이 하한선 대비 +0.047 뿐이고 A6 는 하한선보다
낮은 건 사실이지만, 증강·백본이 바뀌면 **혼동 양상도 같이 바뀝니다.**
기준이 흔들리는 상태에서 클래스를 합치거나 데이터를 지우면 되돌리기 어렵습니다.

📖 [`docs/멘토링_피드백.md`](../docs/멘토링_피드백.md) ·
[`docs/results/STEP4A_베이스라인_실측.md`](../docs/results/STEP4A_베이스라인_실측.md)
